# 🦯 Mushir — AI Mobility Assistant: YOLO Model Training & TFLite Export
**See Beyond Sight. Move with Confidence.**

This Google Colab / Kaggle notebook trains and fine-tunes a real-time object detection model (YOLOv8n) on the **21-class Indian Footpath & Mobility Hazard dataset** (`mushir.yaml`).

### Pipeline Steps:
1. **Check GPU Environment** (Free T4 or V100 GPU)
2. **Install Ultralytics & Mobile Export Tools**
3. **Setup Dataset (`mushir.yaml`)**
4. **Train with Walking-Camera Augmentations**
5. **Evaluate Safety-Critical Per-Class Recall** (Potholes, Open Manholes, Poles, Auto-rickshaws)
6. **Export INT8 Quantized TFLite Model**
7. **Download `.tflite` directly to your phone / Flutter project**

In [ ]:
# Step 1: Verify Free GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics roboflow opencv-python-headless

import ultralytics
ultralytics.checks()

In [ ]:
# Step 3: Clone Mushir Repository or Setup mushir.yaml
!git clone https://github.com/ddevi69105-droid/Mushir.git
%cd Mushir

In [ ]:
# Step 4: Validate Dataset Split and Class Distribution
!python scripts/prepare_dataset.py --validate datasets/mushir

In [ ]:
# Step 5: Fine-tune YOLOv8n with Walking-Camera Augmentations
from ultralytics import YOLO

# Load lightweight baseline pretrained on COCO
model = YOLO('yolov8n.pt')

# Train on Indian Footpath & Mobility Hazards
results = model.train(
    data='datasets/mushir.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    # Augmentations for walking camera tilt, glare, and clutter
    degrees=7.0,
    translate=0.1,
    scale=0.35,
    perspective=0.0004,
    fliplr=0.5,
    mosaic=0.8,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    project='runs/detect',
    name='mushir_colab',
    save=True
)

In [ ]:
# Step 6: Validate Per-Class Recall for Blind Safety
!yolo detect val model=runs/detect/mushir_colab/weights/best.pt data=datasets/mushir.yaml split=test

In [ ]:
# Step 7: Export to INT8 Quantized TFLite for Flutter Android
best_weights = 'runs/detect/mushir_colab/weights/best.pt'
trained_model = YOLO(best_weights)

# Export to TFLite (INT8 quantization for maximum mobile speed)
tflite_path = trained_model.export(format='tflite', imgsz=640, int8=True, nms=False)
print("TFLite Exported to:", tflite_path)

In [ ]:
# Step 8: Download the .tflite model file to your PC/Phone
from google.colab import files
import glob

tflite_files = glob.glob('runs/detect/mushir_colab/weights/*tflite*/*.tflite') + glob.glob('runs/detect/mushir_colab/weights/*.tflite')
if tflite_files:
    print("Downloading:", tflite_files[0])
    files.download(tflite_files[0])
else:
    print("Model file ready in runs/detect/mushir_colab/weights/")